# Exercise 04 — Diversification

MSc Finance · Investments · FHNW · Autumn 2026

Lecture 04 made two claims about diversification and proved them for two assets: risk
falls whenever $\rho < 1$, and the fall stops at a floor set by the average covariance.
This exercise runs both claims on twenty stocks, and then builds an artificial market in
which the floor is known in advance, so you can see where it comes from rather than
inferring it from a curve.

**How to work with this notebook.** The task text is here and on the exercise sheet. Each
code cell is a stub: the `# TODO` lines are the steps, in order. The setup and data cells
below are complete — run them first and leave them alone.

Run **Runtime → Restart and run all** before you trust any number in here. Tasks 2 and 3
draw random numbers, so a cell run on its own picks up wherever the generator happens to
be; a clean top-to-bottom run gives everyone in the room the same figures.

**Data.** Twenty SPI constituents, monthly total-return index levels in Swiss francs,
December 2004 to December 2025 — 253 month-ends and 252 monthly returns. Prices are from
S&P Capital IQ (dividend- and split-adjusted); the `RiskFree` sheet holds the SARON from
the [Swiss National Bank data portal](https://data.snb.ch) and is not needed this week.

Three caveats sit in the `Info` sheet and one of them matters for every number you are
about to compute. The twenty names are stocks that are *still* in the SPI and have traded
continuously since 2004, so anything that was delisted, acquired or failed is missing and
the returns are biased upward. There is deliberately no SPI series in the file: the
equally weighted portfolio of the twenty is the market proxy, so that the proxy and its
constituents carry the same selection bias.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, ASSET_CYCLE, setup_style, load_returns, save_results
setup_style()

SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_04/data/swiss_equities_monthly.xlsx"

# Monthly total-return index levels -> monthly returns.
rets = load_returns(DATA_URL, sheet="Prices", index_col="Date", to_returns=True)

# The Info sheet is a two-column field/value list; the ticker rows are the tail of it.
info = pd.read_excel(DATA_URL, sheet_name="Info")
tick = info[info["Field"].astype(str).str.startswith("SWX:")]
NAME = dict(zip(tick["Field"], tick["Value"]))

N_ASSETS = rets.shape[1]
print(f"{rets.shape[0]} monthly returns, {N_ASSETS} stocks, "
      f"{rets.index.min():%b %Y} to {rets.index.max():%b %Y}")
print(f"{N_ASSETS * (N_ASSETS - 1) // 2} distinct pairs")

## Task 1 — Twenty stocks instead of two

Slide 20 put the volatility of a two-asset portfolio next to the weighted average of the
two individual volatilities and called the gap the diversification benefit: 16 % against
13.11 %, so 2.89 percentage points. Do the same with twenty stocks.

With equal weights $w_i = 1/N$ the weighted average of the individual volatilities is
simply their mean $\bar{\sigma}$, so the comparison is between $\bar{\sigma}$ and

$$\sigma_P = \sqrt{\mathbf{w}' \Sigma\, \mathbf{w}}, \qquad \mathbf{w} = (1/N, \dots, 1/N)'.$$

The returns are monthly; annualise volatilities with $\sqrt{12}$ and the covariance matrix
with 12. Compute four numbers: the average pairwise correlation $\bar{\rho}$ over the 190
distinct pairs, the average annualised volatility $\bar{\sigma}$ of a single stock,
$\sigma_P$ for the equally weighted portfolio of all twenty, and the systematic floor
$\bar{\sigma}\sqrt{\bar{\rho}}$. Package them in a function `risk_summary(r)` — task 4
reuses it on subsamples.

Report the diversification benefit $\bar{\sigma} - \sigma_P$ in percentage points against
the 2.89 pp of the two-asset example, and identify the most and the least correlated pair
in the sample by name.

*Deliverable: the four numbers, the benefit in percentage points, and the two extreme
pairs.*

In [ ]:
# SOLUTION
ANNUAL = np.sqrt(12)                                                          # KEEP

def risk_summary(r):                                                          # KEEP
    n = r.shape[1]                                                            # KEEP
    upper = np.triu(np.ones((n, n), dtype=bool), 1)                           # KEEP

    # STUB: rho_bar, the mean of the n(n-1)/2 distinct pairwise correlations
    rho_bar = r.corr().values[upper].mean()

    # STUB: sd_bar, the mean annualised volatility of a single stock
    sd_bar = (r.std(ddof=1) * ANNUAL).mean()

    # STUB: sigma_P, the annualised volatility of the 1/N portfolio, sqrt(w' Sigma w)
    w = np.repeat(1 / n, n)
    sigma_P = float(np.sqrt(w @ (r.cov().values * 12) @ w))

    # STUB: return the three, plus the systematic floor sd_bar * sqrt(rho_bar)
    return pd.Series({"rho_bar": rho_bar, "sd_bar": sd_bar,
                      "sigma_P": sigma_P, "floor": sd_bar * np.sqrt(rho_bar)})


s = risk_summary(rets)                                                        # KEEP
print(s.to_string(float_format="{:.4f}".format))                              # KEEP
print(f"\ndiversification benefit  sd_bar - sigma_P = "                       # KEEP
      f"{100 * (s['sd_bar'] - s['sigma_P']):.2f} pp")                         # KEEP

In [ ]:
# SOLUTION
upper = np.triu(np.ones((N_ASSETS, N_ASSETS), dtype=bool), 1)                 # KEEP

# STUB: the 190 pairwise correlations as one series, then its range and its extremes
pairs = rets.corr().where(upper).stack().dropna()

print(f"pairwise correlations from {pairs.min():.2f} to {pairs.max():.2f}")   # KEEP
for label, key in [("lowest", pairs.idxmin()), ("highest", pairs.idxmax())]:  # KEEP
    print(f"  {label:8s} {pairs[key]:.2f}   {NAME[key[0]]} / {NAME[key[1]]}") # KEEP

<!-- solution -->
$\bar{\rho} = 0.345$, $\bar{\sigma} = 23.23\ \%$, $\sigma_P = 14.59\ \%$, floor
$= 13.64\ \%$. The diversification benefit is **8.64 percentage points**, three times the
2.89 pp of the two-asset example on slide 20 — the same mechanism, applied nineteen more
times.

The two extreme pairs are worth naming because they are exactly what slide 5 describes.
The highest correlation in the sample is Richemont and Swatch at 0.72: two Geneva-listed
luxury houses selling watches to the same customers through the same channels, so almost
nothing that moves one leaves the other alone. The lowest is Roche and Schindler at 0.07:
a pharmaceutical group and a lift manufacturer share a currency and a business cycle and
very little else. Adding Swatch to a portfolio that already holds Richemont buys you
almost no diversification; adding Schindler to one holding Roche buys you nearly all of
it.

Two mistakes are worth having made once. The first is averaging the whole correlation
matrix instead of its off-diagonal: the twenty ones on the diagonal pull the answer to
0.378, and since the floor moves with $\sqrt{\bar{\rho}}$ that is a floor of 14.3 % rather
than 13.6 %. The second is annualising the covariance matrix with $\sqrt{12}$ instead of
12 — variances scale with time, volatilities with its square root, and the two rules are
one line apart in the same cell.

One number in the output is the reason task 2 exists. $\sigma_P = 14.59\ \%$ against a
floor of 13.64 % says that twenty stocks have *not* exhausted the diversifiable risk;
0.95 pp of it is still sitting in the portfolio. The equally weighted portfolio of all
twenty is not the end of the curve, only the last point we can compute from this file.

## Task 2 — The diversification curve

Slide 8 describes an experiment: draw $N$ stocks at random, form the equally weighted
portfolio, record its volatility, repeat. Slide 9 reports what Statman found running it on
U.S. data — 49 % for a single stock falling to about 19–20 % and flattening there. Run the
experiment on the Swiss sample.

For every $N$ from 1 to 20, draw 500 random subsets of size $N$ without replacement with
`rng.choice(n, size=N, replace=False)`, compute the annualised volatility of the equally
weighted portfolio of each subset, and average over the 500. Plot the resulting curve
against $N$ and mark the floor from task 1.

Then read three things off it. First, $\sigma$ at $N = 1, 5, 10$ and $20$. Second, the
share of the risk reduction available within twenty stocks that the first five deliver,
$(\sigma_1 - \sigma_5)/(\sigma_1 - \sigma_{20})$. Third, the same share measured against
the floor instead of against $N = 20$. The two shares are not the same number, and the
gap between them is the point.

*Deliverable: one figure, the four volatilities, and the two shares.*

In [ ]:
# SOLUTION
# STUB: annualised volatility of the equally weighted portfolio of `cols`
def portfolio_sd(cov, cols):                                                  # KEEP
    w = np.repeat(1 / len(cols), len(cols))
    return float(np.sqrt(w @ cov[np.ix_(cols, cols)] @ w))


def diversification_curve(cov, draws=500):                                    # KEEP
    n = cov.shape[0]                                                          # KEEP
    # STUB: for each N from 1 to n, draw `draws` random subsets of size N without
    # STUB: replacement and average portfolio_sd over them
    out = {N: np.mean([portfolio_sd(cov, rng.choice(n, size=N, replace=False))
                       for _ in range(draws)])
           for N in range(1, n + 1)}
    return pd.Series(out, name="sigma")                                       # KEEP


COV = rets.cov().values * 12                                                  # KEEP
curve = diversification_curve(COV)                                            # KEEP
print(curve.loc[[1, 5, 10, 20]].to_string(float_format="{:.4f}".format))      # KEEP

In [ ]:
# SOLUTION
fig1, ax = plt.subplots(figsize=(8.2, 4.6))                                   # KEEP

# STUB: the curve, and the systematic floor as a horizontal reference line
ax.plot(curve.index, curve.values, marker="o", ms=4, lw=1.6,
        color=FHNW["blue"], label="average volatility of the 1/N portfolio")
ax.axhline(s["floor"], color=FHNW["red"], ls="--", lw=1.3,
           label=f"systematic floor {s['floor']:.1%}")

ax.set_xlabel("Number of stocks N")                                           # KEEP
ax.set_ylabel("Annualised volatility")                                        # KEEP
ax.set_xticks(range(0, 21, 5))                                                # KEEP
ax.set_ylim(0.10, 0.25)                                                       # KEEP
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))                        # KEEP
ax.legend()                                                                   # KEEP
plt.show()                                                                    # KEEP

# STUB: the share of the achievable reduction that the first five stocks deliver,
# STUB: measured once against N = 20 and once against the floor
drop_5 = curve[1] - curve[5]
print(f"first five, against N = 20   {drop_5 / (curve[1] - curve[20]):.1%}")
print(f"first five, against the floor {drop_5 / (curve[1] - s['floor']):.1%}")

<!-- solution -->
The curve runs 23.28 %, 16.46 %, 15.20 %, 14.59 % at $N = 1, 5, 10, 20$. It has the shape
of slide 8 and the shape of Statman's figure, at a lower level throughout: 23 % for a
single Swiss large cap against 49 % for a randomly drawn U.S. stock. That gap is mostly
the sample. These are twenty of the largest companies in Switzerland, all of which
survived to 2025; Statman drew from the whole listed U.S. universe, small caps and
failures included.

The two shares are the substance of the task. Measured against $N = 20$ the first five
stocks deliver **78.5 %** of the reduction, which is the reassuring version and the one
usually quoted. Measured against the floor they deliver **70.8 %**, and the difference
matters because the denominators differ: $\sigma_1 - \sigma_{20}$ is what twenty stocks
happen to have achieved, while $\sigma_1 - \bar{\sigma}\sqrt{\bar{\rho}}$ is what an
infinitely large portfolio could achieve. Dividing by the smaller denominator flatters
every portfolio in the sample, and the flattery grows with how far $N = 20$ still sits
above the floor.

The mistake to expect here is a shrinking one. At $N = 1$ the average over 500 draws is
23.28 % while $\bar{\sigma}$ from task 1 is 23.23 %: same quantity, estimated from 500
draws with replacement across subsets rather than from all twenty stocks, so it wobbles in
the second decimal. At $N = 20$ there is only one possible subset and the average is
exactly the 14.59 % of task 1 — 500 draws of the same portfolio. Sampling noise is largest
where the subsets are smallest, which is also where the curve is steepest, so it is
invisible in the figure and visible in the table.

## Task 3 — Where the floor comes from

The curve in task 2 flattens above a floor it never reaches, and slide 33 says what sets
that floor: the average covariance, which is $\bar{\rho}\,\bar{\sigma}^2$ when every stock
has the same volatility. Rather than infer that from the data, build a market in which it
is true by construction and watch the same curve come out.

Start from $\bar{\sigma}$ and $\bar{\rho}$ of task 1 and write down the equicorrelation
covariance matrix of twenty assets: volatility $\bar{\sigma}$ for every asset, correlation
$\bar{\rho}$ for every pair. Simulate 252 monthly returns from it with
`rng.multivariate_normal` — the matrix has to be in monthly units, so divide the annual
one by 12 — estimate the sample covariance matrix of the simulated returns, and run
`diversification_curve` on it. Plot it on top of the empirical curve. Twenty real Swiss
stocks and twenty artificial assets that share nothing but two numbers should land close
to each other; report how close, and run `risk_summary` on the simulated returns as well.
You put $\bar{\sigma}$ and $\bar{\rho}$ into the simulation and 252 months came back out:
the second call tells you what the sample gives you back, and the difference between the
two curves is not a modelling failure but that.

Then vary the correlation. Hold $\bar{\sigma}$ fixed and repeat for
$\rho \in \{0,\ 0.15,\ 0.35,\ 0.6\}$, drawing the four curves and their four floors
$\bar{\sigma}\sqrt{\rho}$ in one figure.

Finally answer the question the figure poses. For an equicorrelation market the curve has
a closed form,

$$\sigma_P(N) = \bar{\sigma}\,\sqrt{\rho + \frac{1-\rho}{N}},$$

which you can invert. For each of the four correlations, how many stocks are needed to
come within one percentage point of the floor? Report the four values of $N$ next to the
four floors, and be ready to say which of the two columns an investor should care about.

*Deliverable: the overlay figure, the four-curve figure, and a table of four floors and
four values of N.*

In [ ]:
# SOLUTION
# STUB: annualised covariance matrix -- rho off the diagonal, 1 on it, scaled by sd^2
def equicorrelation_cov(sd, rho, n=20):                                       # KEEP
    corr = np.full((n, n), rho)
    np.fill_diagonal(corr, 1.0)
    return corr * sd**2


# STUB: T monthly draws from a zero-mean normal with cov_annual / 12
def simulate(cov_annual, T=252):                                              # KEEP
    return rng.multivariate_normal(np.zeros(cov_annual.shape[0]),
                                   cov_annual / 12, size=T)


SD_BAR, RHO_BAR = s["sd_bar"], s["rho_bar"]                                   # KEEP

# STUB: one simulated market at (SD_BAR, RHO_BAR), then the same curve on its sample cov
sim = simulate(equicorrelation_cov(SD_BAR, RHO_BAR))
curve_sim = diversification_curve(np.cov(sim, rowvar=False) * 12)

fig2, ax = plt.subplots(figsize=(8.2, 4.6))                                   # KEEP
ax.plot(curve.index, curve.values, marker="o", ms=4, lw=1.6,                  # KEEP
        color=FHNW["blue"], label="20 Swiss stocks")                          # KEEP
ax.plot(curve_sim.index, curve_sim.values, marker="s", ms=4, lw=1.6,          # KEEP
        color=FHNW["orange"], label="simulated equicorrelation market")       # KEEP
ax.axhline(s["floor"], color=FHNW["red"], ls="--", lw=1.3,                    # KEEP
           label=f"systematic floor {s['floor']:.1%}")                        # KEEP
ax.set_xlabel("Number of stocks N")                                           # KEEP
ax.set_ylabel("Annualised volatility")                                        # KEEP
ax.set_xticks(range(0, 21, 5))                                                # KEEP
ax.set_ylim(0.10, 0.25)                                                       # KEEP
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))                        # KEEP
ax.legend()                                                                   # KEEP
plt.show()                                                                    # KEEP

# STUB: the largest gap between the two curves, and what the simulated sample
# STUB: gives back for the two parameters that went into it
print(f"largest gap between the two curves "
      f"{100 * (curve - curve_sim).abs().max():.2f} pp\n")
print(pd.DataFrame({"put in": [RHO_BAR, SD_BAR],
                    "came back out": risk_summary(pd.DataFrame(sim))
                                     [["rho_bar", "sd_bar"]].values},
                   index=["rho_bar", "sd_bar"])
      .to_string(float_format="{:.4f}".format))

In [ ]:
# SOLUTION
RHOS = [0.0, 0.15, 0.35, 0.6]                                                 # KEEP
rows = []                                                                     # KEEP
fig3, ax = plt.subplots(figsize=(8.2, 4.6))                                   # KEEP

for rho, colour in zip(RHOS, ASSET_CYCLE):                                    # KEEP
    # STUB: simulate a market with this rho, run the curve, draw it and its floor
    c = diversification_curve(np.cov(simulate(equicorrelation_cov(SD_BAR, rho)),
                                     rowvar=False) * 12)
    floor = SD_BAR * np.sqrt(rho)
    ax.plot(c.index, c.values, lw=1.7, color=colour, label=f"rho = {rho:.2f}")
    ax.axhline(floor, ls=":", lw=1.0, color=colour)

    # STUB: invert sigma_P(N) = sd_bar * sqrt(rho + (1 - rho) / N) at floor + 1 pp
    target = (floor + 0.01) / SD_BAR
    n_within = (1 - rho) / (target**2 - rho)

    rows.append({"rho": rho, "floor": floor, "sigma(N=20)": c[20],            # KEEP
                 "N within 1 pp": np.ceil(n_within)})                         # KEEP

ax.set_xlabel("Number of stocks N")                                           # KEEP
ax.set_ylabel("Annualised volatility")                                        # KEEP
ax.set_xticks(range(0, 21, 5))                                                # KEEP
ax.set_ylim(0, 0.26)                                                          # KEEP
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))                        # KEEP
ax.legend(ncol=2)                                                             # KEEP
plt.show()                                                                    # KEEP

print(pd.DataFrame(rows).to_string(index=False,                               # KEEP
                                   float_format="{:.4f}".format))             # KEEP

<!-- solution -->
The two curves in the overlay never separate by more than 0.59 pp, and they have the same
shape everywhere. Twenty artificial assets that were told nothing about Switzerland except
$\bar{\sigma} = 23.2\ \%$ and $\bar{\rho} = 0.345$ reproduce the empirical curve over its
whole range. That is the result to take from this task: the diversification curve is not a
fact about Swiss industry structure, it is arithmetic on two numbers. Everything that
makes Richemont Richemont and Roche Roche has already been averaged away by the time the
curve is drawn.

The gap that remains is not the model failing, and the second table says so. The
simulation was handed $\bar{\rho} = 0.3449$ and $\bar{\sigma} = 23.23\ \%$; the 252 months
it produced give those back as **0.3709** and **23.75 %**. Feed the returned pair into
$\sigma_P(20) = \bar{\sigma}\sqrt{\bar{\rho} + (1-\bar{\rho})/20}$ and you get 15.1 %,
which is where the orange curve sits. The entire vertical offset is one sample's
estimation error, and 252 monthly observations — twenty-one years, more than most people
ever have — is not enough to pin down an average correlation to better than about three
points. This is the only place in the course where you can check that claim, because it is
the only place where the true value is known rather than estimated.

The four-correlation table is the payoff:

| $\rho$ | floor | $\sigma$ at $N = 20$ | $N$ within 1 pp of the floor |
|---|---|---|---|
| 0.00 | 0.0 % | 5.2 % | 540 |
| 0.15 | 9.0 % | 10.4 % | 25 |
| 0.35 | 13.7 % | 14.8 % | 13 |
| 0.60 | 18.0 % | 18.3 % | 6 |

Read the last column on its own and it says something absurd: the *more* correlated the
market, the *fewer* stocks an investor needs. It is true, and it is not a benefit. High
correlation means the portfolio arrives at its destination quickly because the destination
is close by and terrible. At $\rho = 0.6$ six stocks put you within a percentage point of
18 % volatility and no number of further stocks will ever get you below it. At $\rho = 0$
there is no floor at all — $\sigma_P = \bar{\sigma}/\sqrt{N}$ goes to zero — and the 540
in that row is not a floor calculation but the number of assets it takes to push
volatility below one percentage point outright.

So the two columns answer two different questions. "How many stocks do I need?" is
answered by the last column and is the small question. "How much risk can I get rid of at
all?" is answered by the floor column, and it is the one that decides whether the
portfolio is worth building. Note also that $\sigma$ at $N = 20$ sits *below* the floor in
the $\rho = 0.6$ row: with 252 observations the sample correlation matrix is estimated,
not given, and the 1/N portfolio of one particular draw can land either side of the true
floor. The floor is a property of the population, and it is the estimate that moves.

## Task 4 (optional) — The floor in a crisis

Slide 35 ends with a warning: correlations rise in a downturn, and diversification fails
exactly when it is needed. Test it, and then check what the warning is worth.

Estimate the four numbers of task 1 separately on three windows — the calm run-up
(2005–2007), the crisis (2008–2009), and everything after (2010–2025). `risk_summary` does
the work and the slicing is one line each; note how short the first two windows are, 36
and 24 monthly observations for a matrix with 190 distinct correlations in it.

Slide 35 blames the correlation. But the floor is a product,
$\bar{\sigma}\sqrt{\bar{\rho}}$, and a product has two factors. Decompose the change in
the floor from the calm window to the crisis: once holding $\bar{\rho}$ at its calm value
and letting $\bar{\sigma}$ move to its crisis value, once the other way round. One factor
carries most of the change.

*Deliverable: a three-row table, the two counterfactual floors, and one sentence naming
the factor that does the work.*

In [ ]:
# SOLUTION
WINDOWS = {"2005-2007 calm":   ("2005", "2007"),                              # KEEP
           "2008-2009 crisis": ("2008", "2009"),                              # KEEP
           "2010-2025 after":  ("2010", "2025")}                              # KEEP

# STUB: risk_summary on each window, collected into one table
sub = pd.DataFrame({k: risk_summary(rets.loc[a:b])
                    for k, (a, b) in WINDOWS.items()}).T
print(sub.to_string(float_format="{:.4f}".format))

calm, crisis = sub.loc["2005-2007 calm"], sub.loc["2008-2009 crisis"]         # KEEP

# STUB: the two counterfactual floors -- move one factor at a time
print(f"\nfloor, calm                    {calm['floor']:.2%}")
print(f"  crisis volatility, calm rho  "
      f"{crisis['sd_bar'] * np.sqrt(calm['rho_bar']):.2%}")
print(f"  calm volatility, crisis rho  "
      f"{calm['sd_bar'] * np.sqrt(crisis['rho_bar']):.2%}")
print(f"floor, crisis                  {crisis['floor']:.2%}")

<!-- solution -->
Slide 35 is right about the direction and wrong about the size. $\bar{\rho}$ does rise, from
0.295 in the calm window to 0.407 in the crisis, and it settles at 0.333 afterwards —
higher than before 2008 and lower than during it. But the floor rises from 10.16 % to
25.10 %, and the correlation cannot account for that on its own:

- calm floor: 10.16 %
- crisis volatility, calm correlation: **21.36 %**
- calm volatility, crisis correlation: **11.94 %**
- crisis floor: 25.10 %

Of the 14.94 pp increase, moving the volatility alone produces 11.20 pp and moving the
correlation alone produces 1.78 pp. Volatility does roughly six times the work. That is
arithmetic, not a market observation: $\bar{\sigma}$ enters the floor linearly and
$\bar{\rho}$ enters under a square root, so a doubling of volatility doubles the floor
while a rise in correlation from 0.30 to 0.41 multiplies it by $\sqrt{0.41/0.30} = 1.17$.
Correlations would have to reach 1.0 to move the floor as far as volatility moved it by
itself.

"Correlations go to 1 in a crisis" is therefore a true sentence that misdirects. What
actually happened to a diversified Swiss equity portfolio in 2008 is that everything it
held became much more volatile, and diversification never protected against that in the
first place — it removes the idiosyncratic part, and the crisis was systematic. The
practical consequence for risk budgeting is that a limit written on portfolio volatility
binds in a crisis through $\bar{\sigma}$, and no reallocation across equities relaxes it.
Only a change in exposure does.

The number to be careful with is the sample size. Thirty-six monthly observations for the
calm window and twenty-four for the crisis, against 190 distinct correlations, means these
$\bar{\rho}$ estimates are noisy — although averaging 190 pairs helps far more than
estimating any one of them would suggest. A single pairwise correlation from 24
observations has a standard error of roughly 0.2 and is close to useless; the average of
190 is not, because most of the noise is idiosyncratic to the pair and averages out. That
distinction between an average of noisy estimates and a noisy estimate of an average is
one Lecture 06 returns to.

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or
your own notes.

In [ ]:
# SOLUTION
# STUB: export the curve and the summary tables together with the figures you want
save_results(figures={"diversification_curve": fig1,
                      "simulated_vs_empirical": fig2,
                      "curves_by_correlation": fig3},
             tables={"summary": s.to_frame("value"),
                     "curve": curve.to_frame()},
             name="ex04")

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Every portfolio in this notebook held equal weights. That was a choice, and not obviously
a good one — Nestlé at 13.6 % volatility and UBS at 31.9 % contributed the same share of
capital and very different shares of risk. Lecture 05 drops the constraint and asks which
weights are best, which turns the single curve you plotted into a frontier of them.